# Build district STAAR summary file

This notebook creates a district-level Spring 2026 STAAR summary file.

Inputs:

- `elementary_middle_statewide.csv`: Spring 2026 grades 3-8 STAAR results
- `EOC_statewide.csv`: Spring 2026 end-of-course STAAR results

Output:

- `district_staar_summary.csv`

The final output keeps traditional public districts based on district name suffixes:

- `ISD`
- `CISD`
- `CSD`
- `MSD`(just one case)

Important notes:

- This is based on test administrations, not unique students.
- Rates are calculated from counts, not by averaging TEA's displayed percentages.
- Some districts have grades 3-8 results but no EOC results. This is because a handful of districts do not have high schools.
- Fifth and eight grade science scores are not included. They will be released July 31.
- In some districts, seventh graders took the eight grade math exam and eight graders took Algebra I


In [113]:
import pandas as pd
from pathlib import Path

## File paths

This notebook assumes it is run from the `notebooks/` folder.

Expected repo structure:

```text
staar-demographics/
├── data/
│   ├── raw/
│   │   ├── elementary_middle_statewide.csv
│   │   └── EOC_statewide.csv
│   └── processed/
└── notebooks/
    └── 02_build_staar_summary.ipynb
```


In [114]:
RAW_DATA = Path("../data/raw")
PROCESSED_DATA = Path("../data/processed")

ELEMENTARY_MIDDLE_FILE = RAW_DATA / "elementary_middle_statewide.csv"
EOC_FILE = RAW_DATA / "EOC_statewide.csv"

COMBINED_OUTPUT_FILE = (
    PROCESSED_DATA / "district_staar_summary.csv"
)

ELEMENTARY_MIDDLE_OUTPUT_FILE = (
    PROCESSED_DATA / "district_staar_elementary_middle_summary.csv"
)

EOC_OUTPUT_FILE = (
    PROCESSED_DATA / "district_staar_eoc_summary.csv"
)

PROCESSED_DATA.mkdir(parents=True, exist_ok=True)

## Load STAAR files

The elementary/middle file contains grades 3-8 results.

The EOC file contains high school end-of-course results.


In [115]:
elementary_middle_raw = pd.read_csv(ELEMENTARY_MIDDLE_FILE)
eoc_raw = pd.read_csv(EOC_FILE)

elementary_middle_raw.shape, eoc_raw.shape

((6926, 54), (1135, 53))

In [116]:
elementary_middle_raw.head()

# Charters still included at this point

,Organization,ID/CDC,Administration,Tested Grade,STAAR - Mathematics|Tests Taken,STAAR - Mathematics|Average Scale Score,STAAR - Mathematics|Performance Levels|Did Not Meet|Count,STAAR - Mathematics|Performance Levels|Did Not Meet|Percentage,STAAR - Mathematics|Performance Levels|Approaches and Above|Count,STAAR - Mathematics|Performance Levels|Approaches and Above|Percentage,...,STAAR - Social Studies|Tests Taken,STAAR - Social Studies|Average Scale Score,STAAR - Social Studies|Performance Levels|Did Not Meet|Count,STAAR - Social Studies|Performance Levels|Did Not Meet|Percentage,STAAR - Social Studies|Performance Levels|Approaches and Above|Count,STAAR - Social Studies|Performance Levels|Approaches and Above|Percentage,STAAR - Social Studies|Performance Levels|Meets and Above|Count,STAAR - Social Studies|Performance Levels|Meets and Above|Percentage,STAAR - Social Studies|Performance Levels|Masters|Count,STAAR - Social Studies|Performance Levels|Masters|Percentage
0,A+ ACADEMY,57829,Spring 2026,3,99,1424.0,41.0,41.0,58.0,59.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,A+ ACADEMY,57829,Spring 2026,4,106,1546.0,33.0,31.0,73.0,69.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,A+ ACADEMY,57829,Spring 2026,5,111,1587.0,42.0,38.0,69.0,62.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,A+ ACADEMY,57829,Spring 2026,6,114,1694.0,35.0,31.0,79.0,69.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,A+ ACADEMY,57829,Spring 2026,7,132,1766.0,60.0,45.0,72.0,55.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [117]:
eoc_raw.head()

,Organization,ID/CDC,Administration,STAAR - Algebra I|Tests Taken,STAAR - Algebra I|Average Scale Score,STAAR - Algebra I|Performance Levels|Did Not Meet|Count,STAAR - Algebra I|Performance Levels|Did Not Meet|Percentage,STAAR - Algebra I|Performance Levels|Approaches and Above|Count,STAAR - Algebra I|Performance Levels|Approaches and Above|Percentage,STAAR - Algebra I|Performance Levels|Meets and Above|Count,...,STAAR - U.S. History|Tests Taken,STAAR - U.S. History|Average Scale Score,STAAR - U.S. History|Performance Levels|Did Not Meet|Count,STAAR - U.S. History|Performance Levels|Did Not Meet|Percentage,STAAR - U.S. History|Performance Levels|Approaches and Above|Count,STAAR - U.S. History|Performance Levels|Approaches and Above|Percentage,STAAR - U.S. History|Performance Levels|Meets and Above|Count,STAAR - U.S. History|Performance Levels|Meets and Above|Percentage,STAAR - U.S. History|Performance Levels|Masters|Count,STAAR - U.S. History|Performance Levels|Masters|Percentage
0,A+ ACADEMY,57829,Spring 2026,146,4089.0,19.0,13.0,127.0,87.0,92.0,...,132,4223.0,4.0,3.0,128.0,97.0,103.0,78.0,33.0,25.0
1,A+ UNLIMITED POTENTIAL,101871,Spring 2026,64,3912.0,13.0,20.0,51.0,80.0,19.0,...,8,4185.0,1.0,13.0,7.0,88.0,5.0,63.0,2.0,25.0
2,ABBOTT ISD,109901,Spring 2026,21,4181.0,0.0,0.0,21.0,100.0,15.0,...,23,4197.0,3.0,13.0,20.0,87.0,15.0,65.0,9.0,39.0
3,ABERNATHY ISD,95901,Spring 2026,47,4184.0,2.0,4.0,45.0,96.0,30.0,...,70,4005.0,3.0,4.0,67.0,96.0,32.0,46.0,5.0,7.0
4,ABILENE ISD,221901,Spring 2026,1070,4023.0,194.0,18.0,876.0,82.0,527.0,...,881,4183.0,72.0,8.0,809.0,92.0,569.0,65.0,262.0,30.0


## Keep traditional public districts

The statewide Research Portal files include charters and other organization types.

For this analysis, we keep traditional public districts by suffix:

- `ISD`
- `CISD`
- `CSD`
- `MSD`


In [118]:
district_suffixes_to_keep = (
    " ISD",
    " CISD",
    " CSD",
    " MSD",
)

elementary_middle_raw = elementary_middle_raw[
    elementary_middle_raw["Organization"]
    .str.endswith(district_suffixes_to_keep, na=False)
].copy()

eoc_raw = eoc_raw[
    eoc_raw["Organization"]
    .str.endswith(district_suffixes_to_keep, na=False)
].copy()

print(f"Elementary/middle rows after district filter: {len(elementary_middle_raw):,}")
print(f"EOC rows after district filter: {len(eoc_raw):,}")

print(f"Elementary/middle district IDs after district filter: {elementary_middle_raw['ID/CDC'].nunique():,}")
print(f"EOC district IDs after district filter: {eoc_raw['ID/CDC'].nunique():,}")



Elementary/middle rows after district filter: 6,073
EOC rows after district filter: 995
Elementary/middle district IDs after district filter: 1,018
EOC district IDs after district filter: 995


## Check district coverage

The elementary/middle file can include districts that do not have EOC results. That is expected for districts that do not serve high school grades.


In [119]:
elementary_middle_districts = set(
    elementary_middle_raw["ID/CDC"]
    .dropna()
    .unique()
)

eoc_districts = set(
    eoc_raw["ID/CDC"]
    .dropna()
    .unique()
)

print(f"Elementary/middle districts: {len(elementary_middle_districts):,}")
print(f"EOC districts: {len(eoc_districts):,}")
print(f"Districts in both files: {len(elementary_middle_districts & eoc_districts):,}")
print(f"Districts with grades 3-8 results but no EOC results: {len(elementary_middle_districts - eoc_districts):,}")

Elementary/middle districts: 1,018
EOC districts: 995
Districts in both files: 995
Districts with grades 3-8 results but no EOC results: 23


In [120]:
missing_eoc = elementary_middle_districts - eoc_districts

(
    elementary_middle_raw.loc[
        elementary_middle_raw["ID/CDC"].isin(missing_eoc),
        ["ID/CDC", "Organization"],
    ]
    .drop_duplicates()
    .sort_values("Organization")
)

#Nothing stands out here, these all appear to be small districts 


,ID/CDC,Organization
1695,81906,DEW ISD
1737,133905,DIVIDE ISD
1759,86024,DOSS CONSOLIDATED CSD
2322,166902,GAUSE ISD
2499,90905,GRANDVIEW-HOPKINS ISD
2604,161924,HALLSBURG ISD
2931,19913,HUBBARD ISD
3221,102901,KARNACK ISD
3270,131001,KENEDY COUNTY WIDE CSD
3409,125906,LA GLORIA ISD


In [121]:
# Districts with EOC results but no grades 3-8 row.
#Should be 0

sorted(eoc_districts - elementary_middle_districts)[:50]

[]

## Confirm elementary/middle grades

The elementary/middle file should contain grades 3 through 8. 


In [122]:
sorted(
    elementary_middle_raw["Tested Grade"]
    .dropna()
    .unique()
)

[np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]

## Build elementary/middle STAAR summary

The elementary/middle file contains one row per district-grade combination.

This section sums counts across available grades and subjects in the file.

We use counts instead of TEA's displayed percentages so larger testing groups contribute proportionally to each district's final rate.


In [123]:
# Group the elementary/middle columns by what we need to sum.
# We use counts, not TEA's displayed percentages, so grades/subjects are weighted by the number of tests taken.

em_test_taken_cols = [
    col for col in elementary_middle_raw.columns
    if col.endswith("|Tests Taken")
]

em_approaches_cols = [
    col for col in elementary_middle_raw.columns
    if col.endswith("|Approaches and Above|Count")
]

# Make sure every subject has both a tests-taken column and an approaches-count column.
assert len(em_test_taken_cols) == len(em_approaches_cols)

em_test_taken_cols

['STAAR - Mathematics|Tests Taken',
 'STAAR Spanish - Mathematics|Tests Taken',
 'STAAR - Reading|Tests Taken',
 'STAAR Spanish - Reading|Tests Taken',
 'STAAR - Social Studies|Tests Taken']

In [124]:
# Work from a copy so the raw dataframe stays untouched.
elementary_middle = elementary_middle_raw.copy()

# Convert the count columns to numbers.
# TEA may use blanks for unavailable results; those become NaN and are ignored by sums.
em_count_cols = em_test_taken_cols + em_approaches_cols

for col in em_count_cols:
    elementary_middle[col] = pd.to_numeric(
        elementary_middle[col],
        errors="coerce",
    )

elementary_middle[em_count_cols].describe()

,STAAR - Mathematics|Tests Taken,STAAR Spanish - Mathematics|Tests Taken,STAAR - Reading|Tests Taken,STAAR Spanish - Reading|Tests Taken,STAAR - Social Studies|Tests Taken,STAAR - Mathematics|Performance Levels|Approaches and Above|Count,STAAR Spanish - Mathematics|Performance Levels|Approaches and Above|Count,STAAR - Reading|Performance Levels|Approaches and Above|Count,STAAR Spanish - Reading|Performance Levels|Approaches and Above|Count,STAAR - Social Studies|Performance Levels|Approaches and Above|Count
count,6073.000000,3046.000000,6073.000000,3046.000000,1005.000000,5971.000000,488.000000,5999.000000,618.000000,993.000000
mean,327.045941,12.108339,342.458752,22.226855,362.156219,228.219561,39.116803,267.705618,64.810680,207.725076
std,836.807743,73.645220,858.397970,130.523914,904.023500,611.336240,103.419327,684.309599,181.919198,552.767608
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000
25%,24.000000,0.000000,26.000000,0.000000,28.000000,17.000000,4.000000,21.000000,7.250000,16.000000
50%,68.000000,0.000000,71.000000,0.000000,75.000000,46.000000,9.000000,56.000000,17.000000,41.000000
75%,217.000000,1.000000,230.000000,2.000000,235.000000,147.000000,30.000000,171.000000,50.000000,127.000000
max,12029.000000,2258.000000,11952.000000,3600.000000,10675.000000,9336.000000,1392.000000,9395.000000,2441.000000,5808.000000


In [125]:
# Sum counts across grades and included subjects for each district.

elementary_middle_summary = (
    elementary_middle
    .groupby(
        ["Organization", "ID/CDC", "Administration"],
        as_index=False,
    )
    .apply(
        lambda group: pd.Series({
            "em_tests_taken": group[em_test_taken_cols].sum().sum(),
            "em_approaches_count": group[em_approaches_cols].sum().sum(),
        })
    )
    .reset_index(drop=True)
)

elementary_middle_summary["em_approaches_pct"] = (
    elementary_middle_summary["em_approaches_count"]
    / elementary_middle_summary["em_tests_taken"]
    * 100
)

elementary_middle_summary.sort_values("Organization").head()

/var/folders/5h/mqw74zfn7wz_rq_hgm1kznmr0000gp/T/ipykernel_2956/3308450771.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  elementary_middle


,Organization,ID/CDC,Administration,em_tests_taken,em_approaches_count,em_approaches_pct
0,ABBOTT ISD,109901,Spring 2026,242.0,214.0,88.429752
1,ABERNATHY ISD,95901,Spring 2026,807.0,646.0,80.049566
2,ABILENE ISD,221901,Spring 2026,13079.0,8052.0,61.564340
3,ACADEMY ISD,14901,Spring 2026,1892.0,1476.0,78.012685
4,ADRIAN ISD,180903,Spring 2026,133.0,84.0,63.157895


In [126]:
# Quick checks before moving on to EOC results.

# We should have exactly one row per district-ID-administration combo.
expected_rows = (
    elementary_middle_raw[
        ["Organization", "ID/CDC", "Administration"]
    ]
    .drop_duplicates()
    .shape[0]
)

assert len(elementary_middle_summary) == expected_rows

# Every district should have at least one test administration.
assert (
    elementary_middle_summary["em_tests_taken"] > 0
).all()

# Approaches count should never be larger than tests taken.
assert (
    elementary_middle_summary["em_approaches_count"]
    <= elementary_middle_summary["em_tests_taken"]
).all()

# Percentages should stay in bounds.
assert (
    elementary_middle_summary["em_approaches_pct"]
    .between(0, 100)
).all()

print("All elementary/middle STAAR checks passed.")

All elementary/middle STAAR checks passed.


## Build district-level EOC summary

The EOC file usually has one row per district. Each row contains separate columns for tested EOC subjects such as Algebra I, Biology, English I, English II and U.S. History.

As with the elementary/middle file, we calculate rates from counts rather than averaging percentages.


In [127]:
# Group the EOC columns by what we need to sum.

eoc_test_taken_cols = [
    col for col in eoc_raw.columns
    if col.endswith("|Tests Taken")
]

eoc_approaches_cols = [
    col for col in eoc_raw.columns
    if col.endswith("|Approaches and Above|Count")
]

# Make sure every EOC subject has both a tests-taken column and an approaches-count column.
assert len(eoc_test_taken_cols) == len(eoc_approaches_cols)

eoc_test_taken_cols

['STAAR - Algebra I|Tests Taken',
 'STAAR - Biology|Tests Taken',
 'STAAR - English I|Tests Taken',
 'STAAR - English II|Tests Taken',
 'STAAR - U.S. History|Tests Taken']

In [128]:
# Work from a copy so the raw dataframe stays untouched.
eoc = eoc_raw.copy()

# Convert EOC count columns to numbers.
eoc_count_cols = eoc_test_taken_cols + eoc_approaches_cols

for col in eoc_count_cols:
    eoc[col] = pd.to_numeric(
        eoc[col],
        errors="coerce",
    )

eoc[eoc_count_cols].describe()

,STAAR - Algebra I|Tests Taken,STAAR - Biology|Tests Taken,STAAR - English I|Tests Taken,STAAR - English II|Tests Taken,STAAR - U.S. History|Tests Taken,STAAR - Algebra I|Performance Levels|Approaches and Above|Count,STAAR - Biology|Performance Levels|Approaches and Above|Count,STAAR - English I|Performance Levels|Approaches and Above|Count,STAAR - English II|Performance Levels|Approaches and Above|Count,STAAR - U.S. History|Performance Levels|Approaches and Above|Count
count,995.000000,995.000000,995.000000,995.000000,995.000000,986.000000,968.000000,977.000000,971.000000,969.000000
mean,411.476382,387.077387,433.474372,419.131658,366.969849,341.992901,372.331612,316.587513,310.731205,351.452012
std,1054.913501,986.605312,1111.392241,1090.408140,950.218588,885.504652,937.430157,810.626217,807.811782,902.687974
min,0.000000,0.000000,0.000000,0.000000,0.000000,3.000000,4.000000,2.000000,2.000000,4.000000
25%,31.000000,29.000000,31.000000,30.000000,28.000000,27.000000,31.000000,26.000000,25.000000,29.000000
50%,79.000000,76.000000,82.000000,78.000000,74.000000,69.500000,75.000000,64.000000,63.000000,73.000000
75%,259.000000,245.000000,278.500000,263.500000,235.000000,217.500000,241.250000,205.000000,189.500000,225.000000
max,13513.000000,11691.000000,13661.000000,13674.000000,12007.000000,11092.000000,10978.000000,9411.000000,9756.000000,11209.000000


In [129]:
# Sum counts across EOC subjects for each district.
# The EOC file is usually already one row per district, so this sums across columns.

eoc_summary = eoc[
    ["Organization", "ID/CDC", "Administration"]
].copy()

eoc_summary["eoc_tests_taken"] = eoc[
    eoc_test_taken_cols
].sum(axis=1)

eoc_summary["eoc_approaches_count"] = eoc[
    eoc_approaches_cols
].sum(axis=1)

eoc_summary["eoc_approaches_pct"] = (
    eoc_summary["eoc_approaches_count"]
    / eoc_summary["eoc_tests_taken"]
    * 100
)

eoc_summary.sort_values("Organization").head()

,Organization,ID/CDC,Administration,eoc_tests_taken,eoc_approaches_count,eoc_approaches_pct
2,ABBOTT ISD,109901,Spring 2026,109,104.0,95.412844
3,ABERNATHY ISD,95901,Spring 2026,302,270.0,89.403974
4,ABILENE ISD,221901,Spring 2026,5147,4087.0,79.405479
6,ACADEMY ISD,14901,Spring 2026,689,623.0,90.420900
8,ADRIAN ISD,180903,Spring 2026,55,51.0,92.727273


In [130]:
#  checks before combining elementary/middle and EOC results.

# We should have exactly one row per district-ID-administration combo.
expected_rows = (
    eoc_raw[
        ["Organization", "ID/CDC", "Administration"]
    ]
    .drop_duplicates()
    .shape[0]
)

assert len(eoc_summary) == expected_rows

# Every district should have at least one EOC test administration.
assert (
    eoc_summary["eoc_tests_taken"] > 0
).all()

# Approaches count should never be larger than tests taken.
assert (
    eoc_summary["eoc_approaches_count"]
    <= eoc_summary["eoc_tests_taken"]
).all()

# Percentages should stay in bounds.
assert (
    eoc_summary["eoc_approaches_pct"]
    .between(0, 100)
).all()

print("All EOC STAAR checks passed.")

All EOC STAAR checks passed.


In [131]:
# Create separate processed files for elementary/middle and EOC results.
# These use the same column names so they can be analyzed easily

elementary_middle_output = (
    elementary_middle_summary[
        [
            "Organization",
            "ID/CDC",
            "Administration",
            "em_tests_taken",
            "em_approaches_count",
            "em_approaches_pct",
        ]
    ]
    .rename(
        columns={
            "Organization": "district",
            "ID/CDC": "tea_district_id",
            "Administration": "administration",
            "em_tests_taken": "tests_taken",
            "em_approaches_count": "approaches_count",
            "em_approaches_pct": "approaches_pct",
        }
    )
    .sort_values("district")
    .reset_index(drop=True)
)

eoc_output = (
    eoc_summary[
        [
            "Organization",
            "ID/CDC",
            "Administration",
            "eoc_tests_taken",
            "eoc_approaches_count",
            "eoc_approaches_pct",
        ]
    ]
    .rename(
        columns={
            "Organization": "district",
            "ID/CDC": "tea_district_id",
            "Administration": "administration",
            "eoc_tests_taken": "tests_taken",
            "eoc_approaches_count": "approaches_count",
            "eoc_approaches_pct": "approaches_pct",
        }
    )
    .sort_values("district")
    .reset_index(drop=True)
)

# TEA district IDs are six-character identifiers.
for dataframe in [elementary_middle_output, eoc_output]:
    dataframe["tea_district_id"] = (
        pd.to_numeric(
            dataframe["tea_district_id"],
            errors="raise",
        )
        .astype(int)
        .astype(str)
        .str.zfill(6)
    )

elementary_middle_output.head(), eoc_output.head()

(        district tea_district_id administration  tests_taken  \
 0     ABBOTT ISD          109901    Spring 2026        242.0   
 1  ABERNATHY ISD          095901    Spring 2026        807.0   
 2    ABILENE ISD          221901    Spring 2026      13079.0   
 3    ACADEMY ISD          014901    Spring 2026       1892.0   
 4     ADRIAN ISD          180903    Spring 2026        133.0   
 
    approaches_count  approaches_pct  
 0             214.0       88.429752  
 1             646.0       80.049566  
 2            8052.0       61.564340  
 3            1476.0       78.012685  
 4              84.0       63.157895  ,
         district tea_district_id administration  tests_taken  \
 0     ABBOTT ISD          109901    Spring 2026          109   
 1  ABERNATHY ISD          095901    Spring 2026          302   
 2    ABILENE ISD          221901    Spring 2026         5147   
 3    ACADEMY ISD          014901    Spring 2026          689   
 4     ADRIAN ISD          180903    Spring 2026

In [132]:
# Check the two separate processed outputs before export.

for label, dataframe in [
    ("Elementary/middle", elementary_middle_output),
    ("EOC", eoc_output),
]:
    assert len(dataframe) > 0
    assert dataframe["tea_district_id"].nunique() == len(dataframe)
    assert dataframe["tests_taken"].gt(0).all()
    assert dataframe["approaches_count"].le(dataframe["tests_taken"]).all()
    assert dataframe["approaches_pct"].between(0, 100).all()

    print(f"{label} output checks passed: {len(dataframe):,} districts.")

Elementary/middle output checks passed: 1,018 districts.
EOC output checks passed: 995 districts.


## Combine elementary/middle and EOC results

Some districts have grades 3-8 STAAR results but no EOC results. We keep those districts and treat missing EOC counts as zero.

This means the final file includes all kept districts with reportable elementary/middle STAAR results.


In [133]:
district_staar = elementary_middle_summary.merge(
    eoc_summary,
    on=["Organization", "ID/CDC", "Administration"],
    how="left",
    validate="one_to_one",
)

# Districts without EOC results have missing values after the merge.
# Replace those with zero so they contribute only their elementary/middle results.
district_staar[
    [
        "eoc_tests_taken",
        "eoc_approaches_count",
    ]
] = district_staar[
    [
        "eoc_tests_taken",
        "eoc_approaches_count",
    ]
].fillna(0)

district_staar["tests_taken"] = (
    district_staar["em_tests_taken"]
    + district_staar["eoc_tests_taken"]
)

district_staar["approaches_count"] = (
    district_staar["em_approaches_count"]
    + district_staar["eoc_approaches_count"]
)

district_staar["approaches_pct"] = (
    district_staar["approaches_count"]
    / district_staar["tests_taken"]
    * 100
)

print(f"Elementary/middle summary rows: {len(elementary_middle_summary):,}")

print(f"Elementary/middle districts: {elementary_middle_summary['ID/CDC'].nunique():,}")

print(f"EOC summary rows: {len(eoc_summary):,}")

print(f"EOC districts: {eoc_summary['ID/CDC'].nunique():,}")

print(f"Rows in combined STAAR file: {len(district_staar):,}")

print(f"Districts in combined STAAR file: {district_staar['ID/CDC'].nunique():,}")

district_staar.head()

Elementary/middle summary rows: 1,018
Elementary/middle districts: 1,018
EOC summary rows: 995
EOC districts: 995
Rows in combined STAAR file: 1,018
Districts in combined STAAR file: 1,018


,Organization,ID/CDC,Administration,em_tests_taken,em_approaches_count,em_approaches_pct,eoc_tests_taken,eoc_approaches_count,eoc_approaches_pct,tests_taken,approaches_count,approaches_pct
0,ABBOTT ISD,109901,Spring 2026,242.0,214.0,88.429752,109.0,104.0,95.412844,351.0,318.0,90.598291
1,ABERNATHY ISD,95901,Spring 2026,807.0,646.0,80.049566,302.0,270.0,89.403974,1109.0,916.0,82.596934
2,ABILENE ISD,221901,Spring 2026,13079.0,8052.0,61.564340,5147.0,4087.0,79.405479,18226.0,12139.0,66.602656
3,ACADEMY ISD,14901,Spring 2026,1892.0,1476.0,78.012685,689.0,623.0,90.420900,2581.0,2099.0,81.325068
4,ADRIAN ISD,180903,Spring 2026,133.0,84.0,63.157895,55.0,51.0,92.727273,188.0,135.0,71.808511


## Clean final output

The final file keeps the district name, TEA district ID, administration, test count, Approaches count and Approaches rate.


In [134]:
district_staar = (
    district_staar[
        [
            "Organization",
            "ID/CDC",
            "Administration",
            "tests_taken",
            "approaches_count",
            "approaches_pct",
        ]
    ]
    .rename(
        columns={
            "Organization": "district",
            "ID/CDC": "tea_district_id",
            "Administration": "administration",
        }
    )
    .sort_values("district")
    .reset_index(drop=True)
)

district_staar.head()

,district,tea_district_id,administration,tests_taken,approaches_count,approaches_pct
0,ABBOTT ISD,109901,Spring 2026,351.0,318.0,90.598291
1,ABERNATHY ISD,95901,Spring 2026,1109.0,916.0,82.596934
2,ABILENE ISD,221901,Spring 2026,18226.0,12139.0,66.602656
3,ACADEMY ISD,14901,Spring 2026,2581.0,2099.0,81.325068
4,ADRIAN ISD,180903,Spring 2026,188.0,135.0,71.808511


In [135]:
# Store the TEA district ID as a six-character string.
# Leading zeros are significant (for example, 015905).

district_staar["tea_district_id"] = (
    pd.to_numeric(
        district_staar["tea_district_id"],
        errors="raise",
    )
    .astype(int)
    .astype(str)
    .str.zfill(6)
)

In [136]:
# Final districtwide STAAR checks.

assert len(district_staar) > 0

assert (
    district_staar["tests_taken"] > 0
).all()

assert (
    district_staar["approaches_count"]
    <= district_staar["tests_taken"]
).all()

assert (
    district_staar["approaches_pct"]
    .between(0, 100)
).all()

print("Districtwide STAAR summary looks good.")

Districtwide STAAR summary looks good.


## Export district STAAR summary

In [137]:
# Export the combined exploratory summary and the two separate summaries.

district_staar.to_csv(
    COMBINED_OUTPUT_FILE,
    index=False,
)

elementary_middle_output.to_csv(
    ELEMENTARY_MIDDLE_OUTPUT_FILE,
    index=False,
)

eoc_output.to_csv(
    EOC_OUTPUT_FILE,
    index=False,
)

print(f"Saved combined STAAR summary: {len(district_staar):,} districts")
print(COMBINED_OUTPUT_FILE)

print(
    f"\nSaved elementary/middle summary: "
    f"{len(elementary_middle_output):,} districts"
)
print(ELEMENTARY_MIDDLE_OUTPUT_FILE)

print(f"\nSaved EOC summary: {len(eoc_output):,} districts")
print(EOC_OUTPUT_FILE)

Saved combined STAAR summary: 1,018 districts
../data/processed/district_staar_summary.csv

Saved elementary/middle summary: 1,018 districts
../data/processed/district_staar_elementary_middle_summary.csv

Saved EOC summary: 995 districts
../data/processed/district_staar_eoc_summary.csv


In [138]:
district_staar.head()

,district,tea_district_id,administration,tests_taken,approaches_count,approaches_pct
0,ABBOTT ISD,109901,Spring 2026,351.0,318.0,90.598291
1,ABERNATHY ISD,095901,Spring 2026,1109.0,916.0,82.596934
2,ABILENE ISD,221901,Spring 2026,18226.0,12139.0,66.602656
3,ACADEMY ISD,014901,Spring 2026,2581.0,2099.0,81.325068
4,ADRIAN ISD,180903,Spring 2026,188.0,135.0,71.808511


In [139]:
# Find districts that share the same name but have different IDs.
# Useful for next notebook

duplicate_names = (
    district_staar["district"]
    .value_counts()
    .loc[lambda s: s > 1]
    .index
)

district_staar[
    district_staar["district"].isin(duplicate_names)
].sort_values(["district", "tea_district_id"])

,district,tea_district_id,administration,tests_taken,approaches_count,approaches_pct
67,BIG SANDY ISD,187901,Spring 2026,695.0,600.0,86.330935
68,BIG SANDY ISD,230901,Spring 2026,874.0,610.0,69.794050
152,CENTERVILLE ISD,145902,Spring 2026,863.0,637.0,73.812283
153,CENTERVILLE ISD,228904,Spring 2026,195.0,143.0,73.333333
158,CHAPEL HILL ISD,212909,Spring 2026,4231.0,2746.0,64.901914
159,CHAPEL HILL ISD,225906,Spring 2026,1258.0,1019.0,81.001590
237,DAWSON ISD,058902,Spring 2026,188.0,129.0,68.617021
236,DAWSON ISD,175904,Spring 2026,806.0,542.0,67.245658
281,EDGEWOOD ISD,015905,Spring 2026,8882.0,4908.0,55.257825
282,EDGEWOOD ISD,234903,Spring 2026,1396.0,1090.0,78.080229


In [140]:
# Compare the distributions of test administrations for the
# elementary/middle and EOC summaries separately.

for label, dataframe in [
    ("Elementary/middle", elementary_middle_output),
    ("EOC", eoc_output),
]:
    print(f"\n{'=' * 60}")
    print(label)
    print("=" * 60)

    print(dataframe["tests_taken"].describe())

    print("\nDistricts meeting minimum test thresholds:")

    for cutoff in [100, 250, 500, 1000, 2500, 5000]:
        print(
            f"{cutoff:>6,} tests: "
            f"{(dataframe['tests_taken'] >= cutoff).sum():>4} districts"
        )

# based on these results, I believe we should stick to a minimum of 500 tests in the grade group



Elementary/middle
count      1018.000000
mean       4454.277014
std       11262.772538
min          10.000000
25%         332.250000
50%         902.000000
75%        2934.500000
max      144192.000000
Name: tests_taken, dtype: float64

Districts meeting minimum test thresholds:
   100 tests:  989 districts
   250 tests:  820 districts
   500 tests:  677 districts
 1,000 tests:  486 districts
 2,500 tests:  285 districts
 5,000 tests:  183 districts

EOC
count      995.000000
mean      2018.129648
std       5186.473595
min          1.000000
25%        150.500000
50%        387.000000
75%       1296.500000
max      64546.000000
Name: tests_taken, dtype: float64

Districts meeting minimum test thresholds:
   100 tests:  822 districts
   250 tests:  618 districts
   500 tests:  432 districts
 1,000 tests:  295 districts
 2,500 tests:  170 districts
 5,000 tests:   99 districts


In [141]:
# Districts with perfect or zero Approaches rates.

district_staar[
    district_staar["approaches_pct"].isin([0, 100])
].sort_values(
    ["approaches_pct", "tests_taken"],
    ascending=[True, False],
)

# These are all super small districts
# I believe for the final analysis, we should only do districts with at least 500 tests?

,district,tea_district_id,administration,tests_taken,approaches_count,approaches_pct
952,VYSEHRAD ISD,143904,Spring 2026,39.0,0.0,0.0
939,VALENTINE ISD,122902,Spring 2026,38.0,0.0,0.0
259,DIVIDE ISD,133905,Spring 2026,28.0,0.0,0.0
753,RAMIREZ CSD,066005,Spring 2026,26.0,0.0,0.0
703,PATTON SPRINGS ISD,063906,Spring 2026,20.0,0.0,0.0
819,SAN VICENTE ISD,022903,Spring 2026,14.0,0.0,0.0
